In [ ]:
%load_ext autoreload
%autoreload 2

import os
import pandas as pd
import matplotlib.pyplot as plt

from utils import filter_out_edge_single_cells, extract_single_cell_samples, make_figure

In [ ]:
# batch name
batch = "Batch1"

# Should returned cells be 'random' or 'representative'
cell_selection_method = "random"

# Which channels would you like to return?
# Dictionary of name in images : name in dataframe
im_dict = {
    "DNA": "DAPI_Painting",
    "Stain": "Stain",
    "Phalloidin": "Phalloidin",
}

# Which treatments would you like to return?
treatments = ['Treatment1', 'Treatment2']

# Which plate would you like to return?
plate = "Plate1"

# What genes would you like to return?

genelist = [
    "Gene1",
    "Gene2",
    "Gene3",
    #"nontargeting",
]

# path to single cell profiles
sc_files_dir = 'data/profiles/Batch1/single_cell/single_cell_bygene_normalized_feature_selected'
file_naming = "Batch1_single_cell_profiles_wholeexperiment_normalized_selected__{plate}__{gene}__{treatment}.csv.gz"

# size of images as read into analysis pipeline
im_size = 1600

# path to barcode.csv
metadata_orig = pd.read_csv("Barcodes.csv")

# path to alignments
alignments = pd.read_csv('outputs/Alignments.csv')
alignments['Metadata_Site'] = alignments['Metadata_Site'].astype(int)

# path to load_data
loaddata = pd.read_csv(f'{plate}_load_data_pipeline9.csv')
loaddata = loaddata.replace('/home/ubuntu/bucket/','', regex=True)

# add path to objects, output during analysis
loaddata['Path_CellObjects'] = f'projects/PROJECT/workspace/analysis/{batch}/{plate}-'+loaddata["Metadata_Well"]+"-"+loaddata["Metadata_Site"].astype(str)+f"/segmentation_masks/Plate_{plate}_Well_"+loaddata["Metadata_Well"]+"_Site_"+loaddata["Metadata_Site"].astype(str)+"_Cells_Objects.tiff"

# location to output figures
outdir = "outputs"
if not os.path.exists(outdir):
    os.makedirs(outdir, exist_ok=True)

In [ ]:
# Figure parameters:
n_cells = 20 # number of cells to return
box_size = 100 # in pixels, size of bounding box to crop around cells
dpi_quality = 300
objects = True  # add cell objects
color = True  # add color composite
bc_nuclei = True  # add nuclei from SBS images
bc_nuclei_name = "Cycle01_DAPI"
percentile = 98  # percentile for brightness adjustment

In [ ]:
for treatment in treatments:
    print(f"Processing treatment: {treatment}")
    for gene in genelist:
        print(f"Making figure for gene {gene}")
        # load the single cell profiles for the specified plate, gene, and treatment
        df = pd.read_csv(os.path.join(
                    sc_files_dir,file_naming.format(
                        plate=plate,
                        gene=gene,
                        treatment=treatment
                    )
                )
            )
        # clean up the dataframe
        df = df.rename(columns={"Metadata_Foci_plate":"Metadata_Plate", "Metadata_Foci_well":"Metadata_Well",'Metadata_Foci_site_location':"Metadata_Site"})
        df['Metadata_Site'] = df['Metadata_Site'].astype(int)
        df["Metadata_Label"] = df["Metadata_Well"] + "-" + df["Metadata_Site"].astype(str)

        # add the alignment columns
        df = df.merge(
                alignments,
                on=["Metadata_Plate", "Metadata_Well","Metadata_Site"],
                how="left",
            )
        df = df.merge(loaddata, on=["Metadata_Plate", "Metadata_Well", 'Metadata_Site'], how="left")

        # filter out cells that are too close to the edge of the image
        df, _ = filter_out_edge_single_cells.edgeCellFilter(
                df, "Nuclei_AreaShape_Center_X", "Nuclei_AreaShape_Center_Y", im_size, box_size / 2
            )
        
        if df.shape[0] > 0:
            (
                df,
                cp_features_analysis,
            ) = extract_single_cell_samples.extract_single_cell_samples(
                df, n_cells, cell_selection_method, platefilter=[plate],
            )
            print(f"Filtered to {len(df)} cells using {cell_selection_method} selection method.")
        else:
            print(f"No cells found for {gene} after filtering.")
            continue

        figure = make_figure.make_figure(gene, df, im_dict, cell_selection_method, box_size,
                        objects, color, bc_nuclei, bc_nuclei_name, percentile
                        )


        figure.savefig(os.path.join(outdir, f"{gene}_{treatment}_{cell_selection_method}_{plate}.png"), dpi=dpi_quality)
        plt.close("all")